# 007 — якоря 005 на очищенном трекинге + правило ближайшей линии
Run All. Используются **те же параметры, что выбраны в 005**, без нового перебора. Это позволяет сравнить изменение входных данных и геометрическое правило. tqdm показывает обработку. Все результаты сохраняются в local_review/experiments/007_clean_anchors; прежние модели не меняются.

Чистка воспроизводится из исходного ATD точно по прежней процедуре. При наличии local_review/tracking_cleanup/001_keeper_line/tracking_keeper_line_flags.csv проверяется совпадение флагов каждого наблюдения. Файл не обязателен для воспроизведения в свежем клоне. Вратарь — крайняя точка на глубине 0–16,5 м от своих ворот. Из близости к мячу и числа наблюдений команд исключаются все is_goalkeeper_proxy и suspect_by_rule. Исходные координаты не исправляются и не удаляются.

Линии последних полевых игроков сохраняются как в прежней чистке (до исключения suspect). При пересечённых или отсутствующих линиях правило не применяется. Если наблюдаемый мяч находится в пределах поля, но вне продольного коридора между линиями, назначаем **команду ближайшей линии**, независимо от длины уверенной серии. Это не всегда та же команда, что следует из половины поля. У D безусловный приоритет; выход мяча за боковую линию не считается свидетельством владения.

Остальное — 005: устойчивые якоря, A—неопределённость—A без перехода, граница по максимуму согласованности, запрет склеивания через D. Метод офлайн. Train и проверочная минута изолированы для выбора якорей; полный прогноз использует будущие наблюдения, но не будущие истинные метки. Публичная проверка уже использовалась многократно, её нельзя считать независимым тестом.

Дополнительно сохраняется статистика кадров и максимальных непрерывных серий H/A/D/X: эталон, 005 и 007 на одинаковых 5 минутах; 005 и 007 на полном матче, по таймам отдельно. Закрытый эталон не используется.


In [ ]:
from pathlib import Path
import sys, json, time, warnings, hashlib, itertools, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
sys.path.insert(0,str(ROOT))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from starter.load_smart_tagging import load as load_smart_tagging
from evaluation import possession_metrics as pm
EXPERIMENT = "007_clean_anchors"
OUT = ROOT / "local_review/experiments" / EXPERIMENT
OUT.mkdir(parents=True,exist_ok=True)
ignore = ROOT / "local_review/.gitignore"
if not ignore.exists(): ignore.write_text("*\n")
CFG = json.loads((ROOT / "sol/001_box5_vote100/config.json").read_text())
GRID = json.loads((ROOT / "sol" / EXPERIMENT / "grid.json").read_text())
N, FPS = GAMES[GAME]["n_video_frames"], GAMES[GAME]["fps"]
PITCH = np.array(GAMES[GAME]["pitch"],dtype=np.float64)
BLOCKS = half_video_blocks(GAME)
PATHS = asset_paths()

def save_json(path, value):
    temp = path.with_suffix(path.suffix+".tmp")
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False))
    temp.replace(path)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

started=time.perf_counter()
print(EXPERIMENT, "| Results:", OUT)


## Исходные данные и неизменённый детектор D

In [ ]:
# Load only ATD positions; raw speed units are inconsistent in the source documentation.
# Читаем только координаты ATD: единицы готовой скорости неоднозначны.
track_teams = team_of_each_track(PATHS["atd_meta"])
players = np.full((N, len(track_teams), 2), np.nan, dtype=np.float64)
ball = np.full((N, 2), np.nan, dtype=np.float64)
normalized_players = np.full((N,len(track_teams),2),np.nan,dtype=np.float64)
read_count = 0
with open(PATHS["atd_raw"]) as handle:
    for i, line in enumerate(tqdm(handle,total=N,desc="Tracking / Трекинг")):
        frame_id, player_text, ball_text = line.strip().split(":")
        assert int(frame_id) == i + 1, "ATD/video alignment mismatch"
        player_values = np.fromstring(player_text.replace(";", ","), sep=",").reshape(-1, 3)
        assert len(player_values) == len(track_teams)
        normalized_players[i] = player_values[:, :2]
        players[i] = player_values[:, :2] * PITCH
        ball[i] = np.fromstring(ball_text, sep=",")[:2] * PITCH
        read_count += 1
assert read_count == N
ball_seen = np.isfinite(ball).all(axis=1)
players_seen = np.isfinite(players).all(axis=2)

# The window is an axis-aligned SQUARE, not a radius-2.5 m circle.
# Окно — КВАДРАТ вдоль осей поля, не круг радиусом 2.5 м.
def box_votes(player_xy, ball_xy, team_ids, width_m):
    inside = (np.isfinite(player_xy).all(axis=2)
              & np.isfinite(ball_xy).all(axis=1)[:, None]
              & (np.abs(player_xy - ball_xy[:, None, :]) <= width_m / 2).all(axis=2))
    home_count = (inside & (team_ids == 0)).sum(axis=1)
    away_count = (inside & (team_ids == 1)).sum(axis=1)
    # +1 means H, -1 means A, 0 means abstention for this frame.
    # +1 — H, -1 — A, 0 — нет голоса в этом кадре.
    votes = np.sign(home_count - away_count).astype(np.int8)
    return home_count, away_count, votes

home_count, away_count, frame_votes = box_votes(players, ball, track_teams, CFG["box_width_m"])
# A narrow synthetic check verifies corners, ties, and missing-ball handling.
# Проверяем на искусственных точках углы квадрата, ничью и отсутствие мяча.
test_players = np.array([[[2.5, 2.5], [2.6, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]])
test_ball = np.array([[0., 0.], [0., 0.], [np.nan, np.nan]])
assert box_votes(test_players, test_ball, np.array([0, 1]), 5)[2].tolist() == [1, 0, 0]

# A set-piece interval provides context only; its team tag is not used as a possession label.
# Интервал стандарта — только контекст; его Team не копируем в метку владения.
set_piece_context = np.zeros(N, dtype=bool)
for event in load_smart_tagging():
    if (event["code"] or "").strip().upper() == "SET PIECES":
        start = max(0, event["start_frame"])
        stop = min(N, event["end_frame"])
        set_piece_context[start:stop] = True
print("ATD and automatic set-piece context loaded; no ground truth accessed.")


In [ ]:
lag = CFG["motion_lag_frames"]
dt = lag / FPS
player_motion = np.full(N, np.nan)
ball_motion = np.full(N, np.nan)

# Calculate motion separately in each half, so halftime never creates a false displacement.
# Движение считаем отдельно внутри таймов, не соединяя их через перерыв.
for _, first, last in BLOCKS:
    stop = last + 1
    speed = np.linalg.norm(players[first + lag:stop] - players[first:stop - lag], axis=2) / dt
    speed[(speed > CFG["max_player_motion_mps"]) | ~np.isfinite(speed)] = np.nan
    enough = np.isfinite(speed).sum(axis=1) >= CFG["minimum_motion_tracks"]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="All-NaN slice encountered", category=RuntimeWarning)
        medians = np.nanmedian(speed, axis=1)
    medians[~enough] = np.nan
    player_motion[first + lag:stop] = medians
    seen = ball_seen[first:stop]
    cumulative = np.r_[0, np.cumsum(seen)]
    continuous = (cumulative[lag + 1:] - cumulative[:-(lag + 1)]) == lag + 1
    ball_speed = np.linalg.norm(ball[first + lag:stop] - ball[first:stop - lag], axis=1) / dt
    ball_speed[~continuous] = np.nan
    ball_motion[first + lag:stop] = ball_speed

outside = ball_seen & ((ball < -CFG["outside_margin_m"]) | (ball > PITCH + CFG["outside_margin_m"])).any(axis=1)
inside_field = ball_seen & (ball >= 0).all(axis=1) & (ball <= PITCH).all(axis=1)

def detect_dead_ball():
    dead = np.zeros(N, dtype=bool)
    # Diagnostic counts record triggers, not ground-truth stopping reasons.
    # Счётчики показывают срабатывания эвристик, а не истинные причины остановок.
    reasons = {"outside": 0, "stationary": 0, "missing_and_slow": 0}
    for _, first, last in BLOCKS:
        state = False
        outside_run = missing_run = stationary_run = slow_run = restart_ball_run = restart_players_run = 0
        for i in range(first, last + 1):
            outside_run = outside_run + 1 if outside[i] else 0
            missing_run = missing_run + 1 if not ball_seen[i] else 0
            stationary_run = stationary_run + 1 if ball_motion[i] <= CFG["stationary_ball_mps"] else 0
            slow_run = slow_run + 1 if player_motion[i] <= CFG["slow_players_mps"] else 0
            restart_ball_run = restart_ball_run + 1 if inside_field[i] and ball_motion[i] >= CFG["restart_ball_mps"] else 0
            restart_players_run = restart_players_run + 1 if player_motion[i] >= CFG["restart_players_mps"] else 0
            if not state:
                stationary_threshold = (CFG["set_piece_stationary_confirm_frames"] if set_piece_context[i]
                                        else CFG["stationary_ball_confirm_frames"])
                slow = slow_run >= CFG["slow_players_confirm_frames"]
                if outside_run >= CFG["outside_confirm_frames"]:
                    state = True
                    reasons["outside"] += 1
                elif slow and stationary_run >= stationary_threshold:
                    state = True
                    reasons["stationary"] += 1
                elif slow and missing_run >= CFG["missing_ball_confirm_frames"]:
                    state = True
                    reasons["missing_and_slow"] += 1
            elif not outside[i] and (restart_ball_run >= CFG["restart_ball_confirm_frames"]
                                    or restart_players_run >= CFG["restart_players_confirm_frames"]):
                state = False
                # Reset evidence so a previous stoppage cannot immediately trigger a new one.
                # Сбрасываем накопленные признаки предыдущей остановки.
                outside_run = missing_run = stationary_run = slow_run = 0
            dead[i] = state
    return dead, reasons

dead, dead_trigger_counts = detect_dead_ball()


## Точное воспроизведение прежней чистки

In [ ]:
# Reproduce the exact existing cleanup rules from original normalized ATD coordinates.
# Воспроизводим прежнюю чистку по исходным нормированным координатам ATD.
teams=track_teams; LENGTH=float(PITCH[0]); SIDE_SECONDS=10
KICKOFFS={1:11185,2:113742}
# Infer defending side from team-wide medians, so one stray extreme cannot flip the sides.
# Определяем сторону по медиане всей команды, чтобы одиночный выброс не перевернул стороны.
sides = {}
side_diagnostics = {}
for half, first, last in BLOCKS:
    start = KICKOFFS[half]
    stop = min(start + SIDE_SECONDS * FPS, last + 1)
    centers = {}
    for team in [0, 1]:
        x = normalized_players[start:stop, teams == team, 0]
        valid_rows = np.isfinite(x).sum(axis=1) >= 2
        assert valid_rows.sum() >= FPS, "Insufficient observations to infer sides"
        centers[team] = float(np.median(np.nanmedian(x[valid_rows], axis=1)))
    assert abs(centers[0] - centers[1]) > 0.05, "Side assignment ambiguous"
    sides[half] = {0: "left", 1: "right"} if centers[0] < centers[1] else {0: "right", 1: "left"}
    side_diagnostics[half] = {"home_defends": sides[half][0], "away_defends": sides[half][1],
                             "evidence_frames": stop-start, "method": "median_team_position",
                             "team_median_gap": abs(centers[0]-centers[1])}

# Only longitudinal depth is checked, not the lateral penalty-area boundaries.
# Проверяем глубину от своих ворот, без ограничения по ширине штрафной.
KEEPER_MAX_DEPTH_M = 16.5

def keeper_and_line(x, visible, left, length=LENGTH):
    """Return per-row keeper flags and the deepest remaining defender line."""
    values = x.copy()
    fill = np.inf if left else -np.inf
    values[~visible] = fill
    candidate = np.argmin(values, axis=1) if left else np.argmax(values, axis=1)
    rows = np.arange(len(values))
    candidate_x = values[rows, candidate]
    depth = candidate_x * length if left else (1 - candidate_x) * length
    # Behind-goal outliers cannot become keepers; the boundary is inclusive.
    # Выбросы за воротами не становятся вратарями; граница 16,5 м включается.
    eligible = visible.any(axis=1) & (depth >= -1e-10) & (depth <= KEEPER_MAX_DEPTH_M + 1e-10)
    keeper = np.zeros_like(visible)
    keeper[rows[eligible], candidate[eligible]] = True
    # Without an eligible keeper, retain the deepest player as a defender.
    # Если вратаря нет в наблюдениях, крайний игрок остаётся защитником.
    values[keeper] = fill
    line = np.min(values, axis=1) if left else np.max(values, axis=1)
    line[(visible.sum(axis=1) - eligible.astype(int)) == 0] = np.nan
    return keeper, line

# Small synthetic cases cover both sides, boundary, missing and single observations.
# Искусственные примеры: обе стороны, граница, пропуски и одиночные наблюдения.
test_x = np.array([[.165, .4], [.2, .4], [.1, np.nan], [.3, np.nan],
                   [np.nan, np.nan], [-.01, .1]])
for left in (True, False):
    coords = test_x if left else 1 - test_x
    k, line = keeper_and_line(coords, np.isfinite(coords), left, length=100)
    assert np.array_equal(k.sum(axis=1), [1, 0, 1, 0, 0, 0])
    expected = np.array([.4, .2, np.nan, .3, np.nan, -.01])
    np.testing.assert_allclose(line, expected if left else 1-expected, equal_nan=True)

keepers = np.zeros_like(players_seen)
last_outfield = np.full((N, 2), np.nan)
keeper_diagnostics = {}
for half, first, last in BLOCKS:
    rows = np.arange(first, last + 1)
    for team in [0, 1]:
        slots = np.flatnonzero(teams == team)
        x = normalized_players[first:last + 1, slots, 0]
        visible = players_seen[first:last + 1, slots]
        left = sides[half][team] == "left"
        keeper, line = keeper_and_line(x, visible, left)
        keepers[np.ix_(rows, slots)] = keeper
        last_outfield[rows, team] = line
        depth = x * LENGTH if left else (1-x) * LENGTH
        assert np.all((depth[keeper] >= -1e-10) & (depth[keeper] <= KEEPER_MAX_DEPTH_M + 1e-10))
        assert np.all(keeper.sum(axis=1) <= 1)
        keeper_diagnostics[f"half_{half}_team_{team}"] = {
            "frames_with_keeper": int(keeper.any(axis=1).sum()),
            "frames_with_players_without_keeper": int((visible.any(axis=1) & ~keeper.any(axis=1)).sum())}

suspect = np.zeros_like(players_seen)
flag_valid = np.zeros_like(players_seen)
for half, first, last in BLOCKS:
    rows = np.arange(first, last + 1)
    for team in [0, 1]:
        slots = np.flatnonzero(teams == team)
        opponent = 1 - team
        line = last_outfield[first:last + 1, opponent, None]
        x = normalized_players[first:last + 1, slots, 0]
        valid = players_seen[first:last + 1, slots] & np.isfinite(line)
        # Strict geometric comparison: no ball-position or halfway-line correction.
        # Строго заданная эвристика: без поправки на мяч и среднюю линию.
        behind = x < line if sides[half][opponent] == "left" else x > line
        flag_valid[np.ix_(rows, slots)] = valid
        suspect[np.ix_(rows, slots)] = valid & behind

# Flags must refer to players_seen points only.
# Флаги ставятся только существующим наблюдениям.
assert not np.any(suspect & ~flag_valid)
assert not np.any(keepers & ~players_seen)
summary = {"keeper_max_depth_m": KEEPER_MAX_DEPTH_M, "keeper_region": "own_goal_to_penalty_line_longitudinal_only",
           "keeper_diagnostics": keeper_diagnostics, "frames": N, "side_inference": side_diagnostics,
           "observed_player_points": int(players_seen.sum()), "observed_ball_points": int(ball_seen.sum()),
           "flagged_player_points": int(suspect.sum()), "valid_flag_checks": int(flag_valid.sum()),
           "raw_points_deleted": 0, "possession_model_executed": False}
print(json.dumps(summary, indent=2))

# If the previous cleaned CSV exists, require EXACT equality of all player flags.
# При наличии прежнего CSV проверяем точное совпадение всех флагов игроков.
clean_csv=ROOT/'local_review/tracking_cleanup/001_keeper_line/tracking_keeper_line_flags.csv'
csv_verified=False
if clean_csv.exists():
    checked=0
    columns=['frame','object_type','track_slot','is_goalkeeper_proxy','suspect_by_rule']
    for chunk in tqdm(pd.read_csv(clean_csv,usecols=columns,chunksize=200000),desc='Verify old cleanup / Проверка старой чистки'):
        chunk=chunk[chunk.object_type=='player']
        rr=chunk.frame.to_numpy(dtype=int);ss=chunk.track_slot.to_numpy(dtype=int)-1
        assert np.array_equal(keepers[rr,ss],chunk.is_goalkeeper_proxy.to_numpy(dtype=bool))
        assert np.array_equal(suspect[rr,ss],chunk.suspect_by_rule.to_numpy(dtype=bool))
        checked+=len(chunk)
    assert checked==int(players_seen.sum())
    csv_verified=True
clean_seen=players_seen&~keepers&~suspect

# The rectangle is the longitudinal corridor between the two last-outfield lines.
# Прямоугольник — продольный коридор между линиями последних полевых игроков.
# Out-of-pitch balls are left to D handling; missing/crossed lines do not assign a team.
# Мяч за пределами поля обрабатывается D; пропуски и перекрещенные линии не дают команды.
outside_owner=np.full(N,3,dtype=np.int8)
line_m=last_outfield*PITCH[0]
inside_pitch=ball_seen&(ball>=0).all(1)&(ball<=PITCH).all(1)
for half,first,last in BLOCKS:
    rows=np.arange(first,last+1)
    left_team=0 if sides[half][0]=='left' else 1
    right_team=1-left_team
    left=line_m[rows,left_team];right=line_m[rows,right_team]
    valid_lines=np.isfinite(left)&np.isfinite(right)&(left<right)
    outside=inside_pitch[rows]&valid_lines&((ball[rows,0]<left)|(ball[rows,0]>right))
    distance_to_line=np.abs(ball[rows,0,None]-line_m[rows])
    nearest_line=np.argmin(np.where(np.isfinite(distance_to_line),distance_to_line,np.inf),axis=1)
    outside_owner[rows[outside]]=nearest_line[outside]
cleanup_report=dict(original_player_observations=int(players_seen.sum()),
    keeper_observations_excluded=int((keepers&players_seen).sum()),
    suspect_observations_excluded=int((suspect&players_seen).sum()),
    total_observations_excluded=int((players_seen&~clean_seen).sum()),
    observations_retained=int(clean_seen.sum()),
    old_csv_flags_verified=csv_verified,
    outside_corridor_frames=int(((outside_owner!=3)&~dead).sum()),
    keeper_rule='deepest observation within longitudinal depth 0..16.5 m',
    outfield_lines='same lines as previous cleanup, before removing suspect flags',
    same_half_and_nearest_line_not_assumed_equivalent=True)
save_json(OUT/'cleanup_report.json',cleanup_report)


## Якоря на оставшихся полевых игроках

In [ ]:
# Compute only aggregate geometric support; no track identity is assumed.
# Геометрические свидетельства без предположения о постоянной личности слота.
distance=np.linalg.norm(players-ball[:,None,:],axis=2)
distance[~clean_seen|~ball_seen[:,None]]=np.inf
nearest=np.column_stack([distance[:,track_teams==t].min(axis=1) for t in (0,1)])
team_counts=np.column_stack([clean_seen[:,track_teams==t].sum(axis=1) for t in (0,1)])
reliable=ball_seen&(team_counts>=3).all(axis=1)
del distance

def runs_of(values):
    starts=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1]
    ends=np.r_[starts[1:],len(values)]
    return zip(starts,ends)

def smooth_evidence(values,window):
    # Offline centered smoothing, strictly inside the current live interval.
    # Офлайн-сглаживание строго внутри текущего игрового интервала.
    ids=np.arange(len(values));left=np.maximum(0,ids-window//2)
    right=np.minimum(len(values),ids+window//2+1)
    sums=np.vstack([np.zeros((1,2)),np.cumsum(values,axis=0)])
    return (sums[right]-sums[left])/(right-left)[:,None]

def choose_boundary(evidence,start,stop,old,new,method):
    # Boundary k means old team before k, new team from k onward.
    # Граница k: старая команда до k, новая начиная с k.
    middle=(start+stop)/2
    if method=='midpoint': return int(np.floor(middle))
    segment=evidence[start:stop]
    if not len(segment) or not np.any(segment): return int(np.floor(middle))
    if method=='consistent':
        old_prefix=np.r_[0.,np.cumsum(segment[:,old])]
        new_prefix=np.r_[0.,np.cumsum(segment[:,new])]
        objective=old_prefix+new_prefix[-1]-new_prefix
        candidates=np.flatnonzero(np.isclose(objective,objective.max(),atol=1e-12,rtol=0))+start
        return int(candidates[np.argmin(abs(candidates-middle))])
    if method=='weighted':
        # Rising support for the new team plus a small contest-proximity term.
        # Рост преимущества новой команды плюс небольшой вес близости обеих команд.
        contrast=evidence[:,new]-evidence[:,old]
        candidates=np.arange(start,stop+1)
        now=np.minimum(candidates,len(evidence)-1)
        prev=np.maximum(candidates-1,0)
        rise=np.maximum(contrast[now]-contrast[prev],0.)
        contested=np.minimum(evidence[now,old],evidence[now,new])
        weight=rise+.1*contested
        if weight.sum()<=1e-12:return int(np.floor(middle))
        return int(np.clip(np.floor(np.dot(candidates,weight)/weight.sum()+.5),start,stop))
    raise ValueError(method)

def fill_live(raw,evidence,min_frames,method,edge_frames=50,forced=None):
    # 0=H, 1=A, 3=X. Confident runs are accepted retrospectively in their entirety.
    # Уверенную серию принимаем целиком после проверки её длины; метод офлайн.
    prediction=np.full(len(raw),3,dtype=np.int8); anchors=[]; gaps=[]
    if not len(raw):return prediction,anchors,gaps
    for a,b in runs_of(raw):
        if raw[a]!=3 and (b-a>=min_frames or (forced is not None and forced[a:b].any())):
            anchors.append((int(a),int(b),int(raw[a])))
            prediction[a:b]=raw[a]
    if not anchors:return prediction,anchors,gaps
    a,b,label=anchors[0];prediction[max(0,a-edge_frames):a]=label
    a,b,label=anchors[-1];prediction[b:min(len(raw),b+edge_frames)]=label
    for left,right in zip(anchors,anchors[1:]):
        _,start,old=left;stop,_,new=right
        if old==new:
            # The main rule: same confident team on both sides implies NO turnover.
            # Главное правило: одинаковые команды по краям означают НЕТ перехода.
            prediction[start:stop]=old
            boundary=None
        else:
            boundary=choose_boundary(evidence,start,stop,old,new,method)
            prediction[start:boundary]=old;prediction[boundary:stop]=new
        gaps.append(dict(start=start,stop=stop,old=old,new=new,boundary=boundary))
    return prediction,anchors,gaps

def predict_interval(cfg,first,stop,collect=False):
    # Crucially, train and validation calls cannot see each other's anchors/evidence.
    # Train и validation не видят якоря и свидетельства соседнего раздела.
    result=np.full(stop-first,3,dtype=np.int8);details=[];anchor_count=0;anchor_frames=0
    for half,hfirst,hlast in BLOCKS:
        a=max(first,hfirst);b=min(stop,hlast+1)
        if a>=b:continue
        states=dead[a:b]
        for local_start,local_stop in runs_of(states):
            x=a+local_start;y=a+local_stop
            if states[local_start]:result[x-first:y-first]=2;continue
            d=nearest[x:y];ok=reliable[x:y]
            raw=np.full(y-x,3,dtype=np.int8)
            for team in (0,1):
                confident=ok&(d[:,team]<=cfg['radius_m'])&(d[:,1-team]>=cfg['radius_m']+cfg['opponent_margin_m'])
                raw[confident]=team
            # The corridor rule is direct evidence, including when a keeper is unobserved.
            # Правило коридора действует непосредственно, даже если вратарь не наблюдается.
            forced=outside_owner[x:y]!=3
            raw[forced]=outside_owner[x:y][forced]
            # Gaussian distance support; these scores are NOT calibrated probabilities.
            # Гауссовы оценки близости НЕ являются калиброванными вероятностями.
            evidence=np.exp(-.5*(d/max(cfg['radius_m'],1.))**2)
            evidence[~ok]=0.
            evidence=smooth_evidence(evidence,cfg['evidence_window_frames'])
            pred,anchors,gaps=fill_live(raw,evidence,cfg['min_confident_frames'],cfg['boundary_method'],forced=forced)
            assert np.array_equal(pred[forced],raw[forced])
            result[x-first:y-first]=pred
            anchor_count+=len(anchors);anchor_frames+=sum(j-i for i,j,t in anchors)
            if collect:
                for gap in gaps:
                    details.append(dict(start_frame=x+gap['start'],end_frame_exclusive=x+gap['stop'],
                        left_team='HA'[gap['old']],right_team='HA'[gap['new']],
                        boundary_frame=None if gap['boundary'] is None else x+gap['boundary'],
                        rule='same_team_no_turnover' if gap['old']==gap['new'] else cfg['boundary_method']))
    return np.array(['H','A','D','X'])[result],dict(anchor_runs=anchor_count,anchor_frames=anchor_frames),details

# Behavioral tests: same-team bridge, exact confident labels, all missing, and changepoint.
# Проверки: одинаковая команда по краям, неизменность якорей, пропуски и граница.
for method in ('midpoint','weighted','consistent'):
    raw=np.array([1,1,3,3,3,1,1]);p,_,_=fill_live(raw,np.zeros((7,2)),2,method)
    assert np.all(p==1)
    raw=np.array([0,0,3,3,1,1]);p,_,_=fill_live(raw,np.zeros((6,2)),2,method)
    assert np.all(p[:2]==0) and np.all(p[-2:]==1)
    assert np.count_nonzero(p[1:]!=p[:-1])==1
    p,_,_=fill_live(np.full(7,3),np.zeros((7,2)),2,method)
    assert np.all(p==3)
e=np.array([[1.,0.],[1.,0.],[1.,0.],[0.,1.],[0.,1.],[0.,1.]])
assert choose_boundary(e,2,4,0,1,'consistent')==3
print('Anchor rules verified')

# A single corridor frame must survive the usual minimum confidence-run length.
# Даже одиночный кадр вне коридора обязан сохранить назначенную команду.
p,_,_=fill_live(np.array([3,1,3]),np.zeros((3,2)),8,'consistent',forced=np.array([False,True,False]))
assert p[1]==1


In [ ]:
gt_path=ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv'
truth_table=pm.read_label_csv(gt_path)
truth=pm.label_codes(truth_table)
public=np.flatnonzero(truth!=pm.X)
assert len(public)==7500 and np.all(np.diff(public)==1)
START,END=int(public[0]),int(public[-1]+1);SPLIT=START+240*FPS

def score(pred,first,stop):
    true=truth[first:stop];codes=pm.label_codes(pred)
    assert len(codes)==len(true)
    cm=pm.scored_confusion(pm.confusion(true,codes));runs=pm.valid_runs(true)
    spell=pm.spell_scores(true,pm.fill_abstentions(codes,runs),runs)
    return dict(macroF1=pm.macro_f1(cm),spellF1=spell['mean_f1'],
        accuracy=float(np.mean(true==codes)),X_fraction=float(np.mean(codes==pm.X)),
        true_possessions=spell['n_gt'],predicted_possessions=spell['n_pred'],
        **{name+'_F1':pm.per_class_prf(cm,k)[2] for k,name in enumerate(['H','A','D'])})

save_json(OUT/'run_metadata.json',dict(grid=GRID,train=[START,SPLIT-1],validation=[SPLIT,END-1],
    split_policy='isolated intervals; no future validation anchors in training',
    same_team_rule='always fill same-team anchored gaps unless separated by D',
    parameters_frozen_from='005 selected_training_config.json',
    cleaned_tracking=cleanup_report,
    edge_extension_frames=50,dead_config=CFG,
    sources={k:sha256(PATHS[k]) for k in ['atd_raw','atd_meta','smart_tagging']},public_gt_sha256=sha256(gt_path)))
keys=list(GRID);configs=[dict(zip(keys,v)) for v in itertools.product(*(GRID[k] for k in keys))]
rows=[]
for i,cfg in enumerate(tqdm(configs,desc='Frozen 005 parameters / Фиксированные параметры 005')):
    pred,stats,_=predict_interval(cfg,START,SPLIT)
    rows.append(dict(candidate=i,**cfg,**score(pred,START,SPLIT),**stats))
    if i%10==0:
        save_json(OUT/'progress.json',dict(stage='grid',done=i+1,total=len(configs)))
        pd.DataFrame(rows).to_csv(OUT/'grid_train.csv',index=False)
table=pd.DataFrame(rows).sort_values(['spellF1','macroF1','candidate'],ascending=[False,False,True])
table.to_csv(OUT/'grid_train.csv',index=False)
selected=configs[int(table.iloc[0].candidate)]
save_json(OUT/'selected_config.json',selected)
print('Selected using training only:',selected)


In [ ]:
# Inspect all three families only AFTER training-based selection. Never reselect on validation.
# Проверяем три семейства ПОСЛЕ выбора по обучению; по проверке победителя не меняем.
family_rows=[]
for method in GRID['boundary_method']:
    row=table[table.boundary_method==method].iloc[0]
    cfg=configs[int(row.candidate)]
    pred,stats,_=predict_interval(cfg,SPLIT,END)
    family_rows.append(dict(method=method,candidate=int(row.candidate),**score(pred,SPLIT,END),**stats))
pd.DataFrame(family_rows).to_csv(OUT/'boundary_methods_validation.csv',index=False)
train,train_stats,_=predict_interval(selected,START,SPLIT)
validation,val_stats,_=predict_interval(selected,SPLIT,END)
labels,full_stats,gaps=predict_interval(selected,0,N,collect=True)
result=pd.DataFrame({'frame':np.arange(N),'label':labels})
error=pm.validate_submission(result,N);assert error is None,error
result.to_csv(OUT/'submission.csv',index=False)
pd.DataFrame({'frame':np.arange(SPLIT,END),'label':validation}).to_csv(OUT/'validation_predictions.csv',index=False)
pd.DataFrame(gaps).to_csv(OUT/'gap_diagnostics.csv',index=False)
# This full-match retrospective result may use anchors AFTER the public window.
# Полный офлайн-прогноз может использовать якоря ПОСЛЕ публичного отрезка.
report=dict(train_isolated=score(train,START,SPLIT),validation_isolated=score(validation,SPLIT,END),
            public_full_context=score(labels[START:END],START,END),
            train_anchors=train_stats,validation_anchors=val_stats,full_anchors=full_stats)
save_json(OUT/'metrics.json',report)
comparison=[]
for name,path in [('001_box5_vote100',ROOT/'local_review/experiments/001_box5_vote100/box5_vote100.csv'),
                  ('005_confident_anchors',ROOT/'local_review/experiments/005_confident_anchors/submission.csv')]:
    if path.exists():
        prior=pm.read_label_csv(path);assert pm.validate_submission(prior,N) is None
        comparison.append(dict(experiment=name,context='prior causal history',**score(prior.label.to_numpy()[SPLIT:END],SPLIT,END)))
comparison.append(dict(experiment=EXPERIMENT,context='isolated offline minute',**report['validation_isolated']))
pd.DataFrame(comparison).to_csv(OUT/'comparison_validation.csv',index=False)
save_json(OUT/'progress.json',dict(stage='completed',grid_candidates=len(configs),
    seconds=time.perf_counter()-started,submission_sha256=sha256(OUT/'submission.csv')))
print(json.dumps(report,indent=2))
print(pd.DataFrame(family_rows).to_string(index=False))


In [ ]:
# Verify every same-team gap and every D barrier in the exported answer.
# Проверяем все мосты одинаковой команды и все барьеры D в сохранённом ответе.
assert np.array_equal(labels=='D',dead)
for gap in gaps:
    a,b=gap['start_frame'],gap['end_frame_exclusive']
    assert not dead[a:b].any()
    if gap['left_team']==gap['right_team']:
        assert np.all(labels[a:b]==gap['left_team'])
    else:
        k=gap['boundary_frame']
        assert a<=k<=b
        assert np.all(labels[a:k]==gap['left_team'])
        assert np.all(labels[k:b]==gap['right_team'])
# Change ALL future geometry: training output must remain identical.
# Меняем всю будущую геометрию: результат обучающего отрезка не должен измениться.
saved=nearest[SPLIT:].copy();saved_outside=outside_owner[SPLIT:].copy()
try:
    nearest[SPLIT:]=np.inf;outside_owner[SPLIT:]=3
    isolated,_,_=predict_interval(selected,START,SPLIT)
    assert np.array_equal(isolated,train)
finally:
    nearest[SPLIT:]=saved;outside_owner[SPLIT:]=saved_outside
report['validation_full_context']=score(labels[SPLIT:END],SPLIT,END)
report['validation_context_changed_frames']=int(np.count_nonzero(labels[SPLIT:END]!=validation))
report['same_team_gaps_filled']=sum(g['left_team']==g['right_team'] for g in gaps)
report['different_team_gaps']=sum(g['left_team']!=g['right_team'] for g in gaps)
report['invariants_verified']=True
save_json(OUT/'metrics.json',report)

forced=(outside_owner!=3)&~dead
assert np.array_equal(labels[forced],np.array(['H','A','D','X'])[outside_owner[forced]])
print('Cleanup, corridor rule, D barriers and split invariants passed')


## Статистика эталона, 005 и 007

In [ ]:
# Count maximal runs of each state, including D; X interrupts H/A runs.
# Считаем максимальные серии каждого состояния, включая D; X разрывает H/A.
prior_table=pm.read_label_csv(ROOT/'local_review/experiments/005_confident_anchors/submission.csv')
prior=prior_table.label.to_numpy()
assert pm.validate_submission(prior_table,N) is None
truth_labels=truth_table.label.to_numpy()

def count_states(label_array,blocks,name,window):
    counts={s:0 for s in ['H','A','D','X']};sequences=counts.copy()
    for first,stop in blocks:
        segment=label_array[first:stop]
        for s in counts:
            hit=segment==s
            counts[s]+=int(hit.sum())
            sequences[s]+=int(np.count_nonzero(hit&np.r_[True,~hit[:-1]])) if len(hit) else 0
    return [dict(model=name,window=window,label=s,frames=counts[s],seconds=counts[s]/FPS,sequences=sequences[s]) for s in counts]

stat_rows=[]
for title,prediction in [('public_ground_truth',truth_labels),('005',prior),('007',labels)]:
    stat_rows+=count_states(prediction,[(START,END)],title,'public_5min')
# Ground truth is not available outside the public window: do not count X as unknown truth there.
# Вне публичного окна эталона нет: не выдаём отсутствие разметки за статистику матча.
blocks=[(first,last+1) for _,first,last in BLOCKS]
for title,prediction in [('005',prior),('007',labels)]:
    stat_rows+=count_states(prediction,blocks,title,'full_match_graded_halves')
pd.DataFrame(stat_rows).to_csv(OUT/'state_counts.csv',index=False)
save_json(OUT/'state_counts.json',stat_rows)
assert sum(r['frames'] for r in stat_rows if r['model']=='public_ground_truth' and r['window']=='public_5min')==7500
summary_lines=['# 007 — очищенный трекинг и правило коридора',
    '', 'Параметры якорей и границ зафиксированы из 005, повторного подбора нет. D оставлен прежним.',
    'Вратари и suspect_by_rule исключены из близости к мячу. Линии используются из прежней чистки. Исходные наблюдения не удалены.',
    'Если мяч внутри поля и вне продольного коридора, ему назначается команда ближайшей линии. Это отдельное предположение о владении, не гарантия. Правило половины поля не используется: оно не всегда эквивалентно ближайшей линии.',
    'Метка D имеет приоритет. При пропуске мяча/линий или пересечении линий принудительной команды нет.',
    '', '## Метрики', '', '| Оценка | spellF1 | macroF1 |', '|---|---:|---:|']
for key in ['train_isolated','validation_isolated','validation_full_context','public_full_context']:
    m=report[key];summary_lines.append(f"| {key} | {m['spellF1']:.4f} | {m['macroF1']:.4f} |")
for window in ['public_5min','full_match_graded_halves']:
    summary_lines+=['','## '+window,'','| Источник | Класс | Кадров | Секунд | Последовательностей |','|---|---|---:|---:|---:|']
    for r in stat_rows:
        if r['window']==window:summary_lines.append(f"| {r['model']} | {r['label']} | {r['frames']} | {r['seconds']:.2f} | {r['sequences']} |")
summary_lines+=['','Последовательность — максимальная непрерывная серия одного класса; D и X разрывают владение. Это сырые метки CSV, без заполнения X оценщиком spellF1. Таймы считаются отдельно.',
    'Полный эталон матча недоступен и не использовался. Публичные метрики не являются независимой оценкой: первые четыре минуты ранее участвовали в выборе параметров 005.']
(OUT/'REVIEW_RU.md').write_text('\n'.join(summary_lines)+'\n')
print(pd.DataFrame(stat_rows).to_string(index=False))
